# Data Quality Check

In [1]:
from pathlib import Path
from collections import Counter
import pandas as pd
import numpy as np
import json
import re
from tqdm.auto import tqdm
from IPython.display import display
import random
import unicodedata
import math
from scipy.stats import hypergeom
from sklearn.model_selection import StratifiedGroupKFold
from itertools import combinations
from scipy.optimize import milp, Bounds, LinearConstraint
from scipy.sparse import lil_matrix

In [2]:
BASE = Path("/home/ftp_246/data_5/多模態2026_ig_top200")

METADATA_DIR = BASE / "Metadata_by_Influencer"
VIDEO_DIR = BASE / "Video_data_fullest/ig_vedios"
JSON_DIR = BASE / "Prompt_Description/Fullset_v20"

PROFILE_PATH = Path(
    "/home/yingtzudai/Multimodal_SPoD_Video/data/Influencer_Profile.csv"
)

# 檢查路徑
paths = {
    "Profile": PROFILE_PATH,
    "Metadata": METADATA_DIR,
    "MP4": VIDEO_DIR,
    "JSON": JSON_DIR
}

for name, path in paths.items():
    print(f"{name}: {path.exists()} | {path}")

if not all(path.exists() for path in paths.values()):
    raise FileNotFoundError("請先修正不存在的資料路徑")

# 載入網紅 Profile，保留所有原始欄位
df_profile = pd.read_csv(
    PROFILE_PATH,
    dtype={"ig_id": "string", "ig_strong_id": "string"}
)

df_profile["ig_id"] = df_profile["ig_id"].str.strip()

print("Profile rows:", len(df_profile))

Profile: True | /home/yingtzudai/Multimodal_SPoD_Video/data/Influencer_Profile.csv
Metadata: True | /home/ftp_246/data_5/多模態2026_ig_top200/Metadata_by_Influencer
MP4: True | /home/ftp_246/data_5/多模態2026_ig_top200/Video_data_fullest/ig_vedios
JSON: True | /home/ftp_246/data_5/多模態2026_ig_top200/Prompt_Description/Fullset_v20
Profile rows: 203


## 讀取所有 Metadata，產生預期影片檔名

In [3]:
def format_modified_time(value):
    """將 modified_time_tw 轉為 YYYYMMDDHHMMSS。"""
    if pd.isna(value):
        return None

    s = str(value).strip()

    # 已經是 14 位數字
    if re.fullmatch(r"\d{14}", s):
        return s

    # 一般日期時間格式
    dt = pd.to_datetime(s, errors="coerce")

    if pd.notna(dt):
        return dt.strftime("%Y%m%d%H%M%S")

    return None


metadata_frames = []
metadata_errors = []

for csv_path in METADATA_DIR.rglob("*.csv"):
    try:
        df = pd.read_csv(
            csv_path,
            dtype={
                "id": "string",
                "media_id": "string",
                "post_owner.username": "string"
            },
            low_memory=False
        )

        required = [
            "id",
            "media_id",
            "post_owner.username",
            "modified_time_tw"
        ]

        missing = [c for c in required if c not in df.columns]

        if missing:
            metadata_errors.append({
                "file": str(csv_path),
                "error": f"Missing columns: {missing}"
            })
            continue

        df["metadata_path"] = str(csv_path)
        metadata_frames.append(df)

    except Exception as e:
        metadata_errors.append({
            "file": str(csv_path),
            "error": str(e)
        })

if not metadata_frames:
    raise ValueError("沒有成功讀取任何 Metadata CSV")

df_metadata = pd.concat(
    metadata_frames, ignore_index=True
)

# 統一欄位名稱
df_metadata = df_metadata.rename(columns={
    "id": "post_id",
    "post_owner.username": "ig_id"
})

for col in ["ig_id", "post_id", "media_id"]:
    df_metadata[col] = df_metadata[col].astype("string").str.strip()

df_metadata["modified_timestamp"] = (
    df_metadata["modified_time_tw"]
    .apply(format_modified_time)
)

# 確保組合檔名所需的欄位完整
valid_name = (
    df_metadata["ig_id"].notna()
    & df_metadata["post_id"].notna()
    & df_metadata["media_id"].notna()
    & df_metadata["modified_timestamp"].notna()
)

df_metadata["expected_video_filename"] = pd.NA

df_metadata.loc[valid_name, "expected_video_filename"] = (
    df_metadata.loc[valid_name, "ig_id"]
    + "-"
    + df_metadata.loc[valid_name, "modified_timestamp"]
    + "-"
    + df_metadata.loc[valid_name, "media_id"]
    + ".mp4"
)

print("Metadata rows:", len(df_metadata))
print("CSV read errors:", len(metadata_errors))
print("Rows with valid filenames:", valid_name.sum())

display(
    df_metadata[
        [
            "ig_id",
            "post_id",
            "media_id",
            "modified_time_tw",
            "expected_video_filename"
        ]
    ].head()
)

Metadata rows: 19443
CSV read errors: 0
Rows with valid filenames: 19443


,ig_id,post_id,media_id,modified_time_tw,expected_video_filename
0,traveggo,916191564143111,916191564143111,2025-08-05 00:37:18+08:00,traveggo-20250805003718-916191564143111.mp4
1,traveggo,3243955289117081,3243955289117081,2025-07-19 18:17:49+08:00,traveggo-20250719181749-3243955289117081.mp4
2,traveggo,1604802410544990,1604802410544990,2025-11-19 06:46:44+08:00,traveggo-20251119064644-1604802410544990.mp4
3,traveggo,849685401437704,849685401437704,2025-02-17 01:26:48+08:00,traveggo-20250217012648-849685401437704.mp4
4,traveggo,1452518233264809,1452518233264809,2025-09-17 14:44:07+08:00,traveggo-20250917144407-1452518233264809.mp4


## 掃描 MP4 與 JSON，建立 Post-level Inventory

In [4]:
# 1. 掃描所有 MP4
mp4_rows = []

for path in VIDEO_DIR.rglob("*"):
    if path.is_file() and path.suffix.lower() == ".mp4":
        mp4_rows.append({
            "ig_id": path.parent.name,
            "video_filename": path.name,
            "video_path": str(path),
            "video_size_mb": round(
                path.stat().st_size / (1024 ** 2), 2
            )
        })

df_mp4 = pd.DataFrame(
    mp4_rows,
    columns=[
        "ig_id", "video_filename",
        "video_path", "video_size_mb"
    ]
)

# 2. 掃描 JSON
json_rows = []
json_errors = []

for path in JSON_DIR.rglob("*.json"):
    try:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        video_name = data.get("videoName")

        if not isinstance(video_name, str):
            video_name = None

        json_rows.append({
            "ig_id": path.parent.name,
            "video_filename": (
                Path(video_name).name if video_name else None
            ),
            "json_path": str(path)
        })

    except (OSError, ValueError, TypeError) as e:
        json_errors.append({
            "file": str(path),
            "error": str(e)
        })

df_json = pd.DataFrame(
    json_rows,
    columns=["ig_id", "video_filename", "json_path"]
)

# 3. 以 ig_id + 完整檔名作為匹配鍵
keys = ["ig_id", "video_filename"]

# 找出重複鍵，避免 merge 造成多對多膨脹
mp4_key_counts = df_mp4.groupby(keys, dropna=False).size()
json_key_counts = df_json.groupby(keys, dropna=False).size()

mp4_unique = df_mp4[
    ~df_mp4.set_index(keys).index.isin(
        mp4_key_counts[mp4_key_counts > 1].index
    )
]

json_unique = df_json[
    ~df_json.set_index(keys).index.isin(
        json_key_counts[json_key_counts > 1].index
    )
]

# 4. 建立貼文清單
df_post_inventory = df_metadata.copy()

df_post_inventory["video_filename"] = (
    df_post_inventory["expected_video_filename"]
)

df_post_inventory = df_post_inventory.merge(
    mp4_unique,
    on=keys,
    how="left",
    validate="many_to_one"
)

df_post_inventory = df_post_inventory.merge(
    json_unique,
    on=keys,
    how="left",
    validate="many_to_one"
)

# 5. 建立完整度旗標
df_post_inventory["has_mp4"] = (
    df_post_inventory["video_path"].notna()
)

df_post_inventory["has_json"] = (
    df_post_inventory["json_path"].notna()
)

df_post_inventory["is_complete"] = (
    df_post_inventory["has_mp4"]
    & df_post_inventory["has_json"]
)

df_post_inventory["match_status"] = np.select(
    [
        df_post_inventory["is_complete"],
        df_post_inventory["has_mp4"],
        df_post_inventory["has_json"]
    ],
    [
        "complete",
        "mp4_only",
        "json_only"
    ],
    default="missing_both"
)

# 6. 重複貼文標記
df_post_inventory["duplicate_post_id"] = (
    df_post_inventory.duplicated(
        ["ig_id", "post_id"], keep=False
    )
)

# 簡化輸出欄位，保留研究所需資訊
post_columns = [
    "ig_id",
    "post_id",
    "media_id",
    "short_code",
    "text",
    "duration",
    "tags",
    "modified_time_tw",
    "video_filename",
    "video_path",
    "json_path",
    "video_size_mb",
    "has_mp4",
    "has_json",
    "is_complete",
    "match_status",
    "duplicate_post_id",
    "metadata_path"
]

df_post_inventory = df_post_inventory[
    [c for c in post_columns if c in df_post_inventory.columns]
]

print("Metadata rows:", len(df_post_inventory))
print("MP4 files:", len(df_mp4))
print("JSON records:", len(df_json))
print("Complete rows:", df_post_inventory["is_complete"].sum())

display(df_post_inventory.head())

Metadata rows: 19443
MP4 files: 19312
JSON records: 19116
Complete rows: 19102


,ig_id,post_id,media_id,short_code,text,duration,tags,modified_time_tw,video_filename,video_path,json_path,video_size_mb,has_mp4,has_json,is_complete,match_status,duplicate_post_id,metadata_path
0,traveggo,916191564143111,916191564143111,DE2RQZzy7ik,#同居觀察日記 Ep1\n同居是感情必修學分\n兩人住進同一個屋簷下\n首先迎來的 絕對是習...,90.02,{'0': '1833997590677795'},2025-08-05 00:37:18+08:00,traveggo-20250805003718-916191564143111.mp4,/home/ftp_246/data_5/多模態2026_ig_top200/Video_d...,/home/ftp_246/data_5/多模態2026_ig_top200/Prompt_...,17.34,True,True,True,complete,False,/home/ftp_246/data_5/多模態2026_ig_top200/Metadat...
1,traveggo,3243955289117081,3243955289117081,DFK56DuvHOw,#同居觀察日記 Ep2\n經歷女生月經週期像換了4個女友……\n\n#情侶日常 #vegan...,31.81,NaN,2025-07-19 18:17:49+08:00,traveggo-20250719181749-3243955289117081.mp4,/home/ftp_246/data_5/多模態2026_ig_top200/Video_d...,/home/ftp_246/data_5/多模態2026_ig_top200/Prompt_...,4.79,True,True,True,complete,False,/home/ftp_246/data_5/多模態2026_ig_top200/Metadat...
2,traveggo,1604802410544990,1604802410544990,DFNcyBZSSvJ,#同居觀察日記 Ep3\n我以為我在女生界洗澡算快了\n同居才發現男女洗澡速度真的差好多😂\...,89.20,{'0': '459340223188257'},2025-11-19 06:46:44+08:00,traveggo-20251119064644-1604802410544990.mp4,/home/ftp_246/data_5/多模態2026_ig_top200/Video_d...,/home/ftp_246/data_5/多模態2026_ig_top200/Prompt_...,16.42,True,True,True,complete,False,/home/ftp_246/data_5/多模態2026_ig_top200/Metadat...
3,traveggo,849685401437704,849685401437704,DFXxOBDv1YN,帥哥帥妹祝大家新年快樂❤️\n2025 蛇麼都順利 平安 健康 心中有愛,15.16,NaN,2025-02-17 01:26:48+08:00,traveggo-20250217012648-849685401437704.mp4,/home/ftp_246/data_5/多模態2026_ig_top200/Video_d...,/home/ftp_246/data_5/多模態2026_ig_top200/Prompt_...,2.81,True,True,True,complete,False,/home/ftp_246/data_5/多模態2026_ig_top200/Metadat...
4,traveggo,1452518233264809,1452518233264809,DGYHxkTvIr9,#同居觀察日記 Ep4 \n啊不是要保護我？？？？？？,40.87,NaN,2025-09-17 14:44:07+08:00,traveggo-20250917144407-1452518233264809.mp4,/home/ftp_246/data_5/多模態2026_ig_top200/Video_d...,/home/ftp_246/data_5/多模態2026_ig_top200/Prompt_...,5.06,True,True,True,complete,False,/home/ftp_246/data_5/多模態2026_ig_top200/Metadat...


## 產生網紅完整統計表

In [5]:
# 1. 以唯一 post_id 計算貼文數
post_stats = (
    df_post_inventory
    .groupby("ig_id", dropna=False)
    .agg(
        metadata_row_count=("post_id", "size"),
        metadata_post_count=("post_id", "nunique"),
        matched_video_count=("has_mp4", "sum"),
        matched_json_count=("has_json", "sum"),
        fully_matched_count=("is_complete", "sum"),
        duplicate_post_rows=("duplicate_post_id", "sum")
    )
    .reset_index()
)

# 2. MP4 實際檔案數
mp4_stats = (
    df_mp4.groupby("ig_id")
    .agg(
        mp4_count=("video_path", "size"),
        total_video_size_mb=("video_size_mb", "sum")
    )
    .reset_index()
)

# 3. JSON 實際檔案數
json_stats = (
    df_json.groupby("ig_id")
    .agg(json_count=("json_path", "size"))
    .reset_index()
)

# 4. 合併 Profile 與所有統計
# 使用 outer，避免遺漏 Profile 以外的帳號
df_influencer_summary = (
    df_profile
    .merge(
        post_stats,
        on="ig_id",
        how="outer",
        indicator="profile_merge_status"
    )
    .merge(mp4_stats, on="ig_id", how="outer")
    .merge(json_stats, on="ig_id", how="outer")
)

count_columns = [
    "metadata_row_count",
    "metadata_post_count",
    "matched_video_count",
    "matched_json_count",
    "fully_matched_count",
    "duplicate_post_rows",
    "mp4_count",
    "json_count"
]

for col in count_columns:
    df_influencer_summary[col] = (
        df_influencer_summary[col]
        .fillna(0)
        .astype(int)
    )

df_influencer_summary["total_video_size_mb"] = (
    df_influencer_summary["total_video_size_mb"]
    .fillna(0)
)

# 5. 計算完整率
denominator = (
    df_influencer_summary["metadata_post_count"]
    .replace(0, np.nan)
)

df_influencer_summary["complete_rate"] = (
    df_influencer_summary["fully_matched_count"]
    / denominator
)

# 6. 標記是否有可用資料
df_influencer_summary["has_usable_video"] = (
    df_influencer_summary["fully_matched_count"] > 0
)

# 7. 將重要欄位放在前面，其餘 Profile 欄位保留
priority_cols = [
    "ig_id",
    "ig_strong_id",
    "AsiaKOL_category",
    "Followers",
    "Following",
    "posts",
    "metadata_post_count",
    "mp4_count",
    "json_count",
    "matched_video_count",
    "matched_json_count",
    "fully_matched_count",
    "complete_rate",
    "has_usable_video",
    "total_video_size_mb"
]

priority_cols = [
    c for c in priority_cols
    if c in df_influencer_summary.columns
]

other_cols = [
    c for c in df_influencer_summary.columns
    if c not in priority_cols
]

df_influencer_summary = (
    df_influencer_summary[priority_cols + other_cols]
    .sort_values(
        "fully_matched_count",
        ascending=False
    )
    .reset_index(drop=True)
)

display(df_influencer_summary)

print("Total influencers:", len(df_influencer_summary))
print(
    "Influencers with complete videos:",
    df_influencer_summary["has_usable_video"].sum()
)
print(
    "Total complete post rows:",
    df_influencer_summary["fully_matched_count"].sum()
)

,ig_id,ig_strong_id,AsiaKOL_category,Followers,Following,posts,metadata_post_count,mp4_count,json_count,matched_video_count,...,has_usable_video,total_video_size_mb,Aisa_Order,person_name,ig_name,ig_url,AsiaKOL_followers,metadata_row_count,duplicate_post_rows,profile_merge_status
0,dr.acne_scar_queen,4833663962,"知識教育,醫療健康",189848,259,2943,639,639,639,639,...,True,4142.59,137,痘疤醫生---莊盈彥醫師,痘疤女王 莊盈彥,https://www.instagram.com/dr.acne_scar_queen/,"922,505",639,0,both
1,mayukichou,18346291,"時尚潮流,高階經理人",309258,1748,8861,538,538,538,538,...,True,2242.52,163,周小葳,葳老闆｜周品均,https://www.instagram.com/mayukichou/,"636,488",538,0,both
2,cocowine0205,2307480066,"美食料理,趣味搞笑",157798,782,1593,391,391,391,391,...,True,5889.88,98,可可酒精,可可酒精,https://www.instagram.com/cocowine0205/,"853,725",391,0,both
3,yga0721,45717781,趣味搞笑,1495461,501,8178,344,344,342,344,...,True,4807.44,1,蔡阿嘎,蔡阿嘎,https://www.instagram.com/yga0721/,"5,893,570",344,0,both
4,ladyflavor_cui,3656815405,NaN,369611,122,1378,342,342,342,342,...,True,4844.68,14,好味小姐,好味小姐脆脆,https://www.instagram.com/ladyflavor_cui/,NaN,342,0,both
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
198,alvin701,1972535324,知識教育,187869,142,1329,1,0,0,0,...,False,0.00,115,艾爾文,艾爾文,https://www.instagram.com/alvin701/,"1,563,768",1,0,both
199,lynnwu0219_2,47642521277,帶貨分潤,76276,93,21,0,0,0,0,...,False,0.00,39,泱泱LYNN,泱泱LYNN,https://www.instagram.com/lynnwu0219_2/,"1,965,591",0,0,left_only
200,skauramomo,515352630,"時尚潮流,表演藝術",0,0,0,35,0,0,0,...,False,0.00,195,紫玥桃,紫玥桃 Cosplay,https://www.instagram.com/skauramomo/,"676,777",35,0,both
201,oldwangwang,3216171228,"趣味搞笑,時事討論",121681,57,289,1,1,1,0,...,False,5.13,177,老王-oldwang,老王 oldwong,https://www.instagram.com/oldwangwang/,"2,501,639",1,0,both


Total influencers: 203
Influencers with complete videos: 196
Total complete post rows: 19102


## 建立集中式 Data Quality Summary

In [6]:
# 重複貼文 ID
duplicate_posts = (
    df_post_inventory["duplicate_post_id"].sum()
)

# 沒有對應到 Metadata 的 MP4
expected_keys = df_metadata[
    ["ig_id", "expected_video_filename"]
].dropna().drop_duplicates().rename(
    columns={"expected_video_filename": "video_filename"}
)

orphan_mp4 = df_mp4.merge(
    expected_keys,
    on=["ig_id", "video_filename"],
    how="left",
    indicator=True
)

orphan_mp4_count = (
    orphan_mp4["_merge"] == "left_only"
).sum()

# 沒有對應到 Metadata 的 JSON
orphan_json = df_json.merge(
    expected_keys,
    on=["ig_id", "video_filename"],
    how="left",
    indicator=True
)

orphan_json_count = (
    orphan_json["_merge"] == "left_only"
).sum()

# Profile 沒有出現的網紅
profile_ids = set(df_profile["ig_id"].dropna())

all_data_ids = (
    set(df_metadata["ig_id"].dropna())
    | set(df_mp4["ig_id"].dropna())
    | set(df_json["ig_id"].dropna())
)

unknown_influencers = all_data_ids - profile_ids

# Profile 中重複的識別碼
duplicate_strong_ids = (
    df_profile["ig_strong_id"]
    .dropna()
    .duplicated()
    .sum()
)

quality_checks = [
    ("Metadata 總列數", len(df_metadata)),
    ("唯一網紅數（所有來源）", len(profile_ids | all_data_ids)),
    ("實際 MP4 數", len(df_mp4)),
    ("成功解析 JSON 數", len(df_json)),
    ("完整匹配貼文列數",
     int(df_post_inventory["is_complete"].sum())),
    ("缺少 MP4 的貼文列數",
     int((~df_post_inventory["has_mp4"]).sum())),
    ("缺少 JSON 的貼文列數",
     int((~df_post_inventory["has_json"]).sum())),
    ("缺少 MP4 與 JSON 的貼文列數",
     int((df_post_inventory["match_status"] == "missing_both").sum())),
    ("重複貼文 ID 列數", int(duplicate_posts)),
    ("重複 ig_strong_id 數", int(duplicate_strong_ids)),
    ("沒有 Metadata 對應的 MP4 數", int(orphan_mp4_count)),
    ("沒有 Metadata 對應的 JSON 數", int(orphan_json_count)),
    ("Profile 以外的網紅數", len(unknown_influencers)),
    ("Metadata CSV 讀取錯誤數", len(metadata_errors)),
    ("JSON 解析錯誤數", len(json_errors)),
    ("重複 MP4 匹配鍵數",
     int((mp4_key_counts > 1).sum())),
    ("重複 JSON 匹配鍵數",
     int((json_key_counts > 1).sum()))
]

df_data_quality = pd.DataFrame(
    quality_checks,
    columns=["check_item", "count"]
)

display(df_data_quality)

# 找出 Metadata 有紀錄，但找不到對應 MP4 的貼文
df_missing_videos = df_post_inventory.loc[
    ~df_post_inventory["has_mp4"]
].copy()

# 重新產生預期檔名，方便查找
df_missing_videos["expected_video_filename"] = (
    df_missing_videos["video_filename"]
)

# 選擇要顯示的欄位
columns = [
    "ig_id",
    "post_id",
    "media_id",
    "modified_time_tw",
    "expected_video_filename",
    "has_json",
    "match_status"
]

df_missing_videos = df_missing_videos[columns]

print(f"Missing MP4 posts: {len(df_missing_videos)}")

# 顯示前 20 筆缺少 MP4 的貼文
display(df_missing_videos.head(20))

,check_item,count
0,Metadata 總列數,19443
1,唯一網紅數（所有來源）,203
2,實際 MP4 數,19312
3,成功解析 JSON 數,19116
4,完整匹配貼文列數,19102
5,缺少 MP4 的貼文列數,133
6,缺少 JSON 的貼文列數,341
7,缺少 MP4 與 JSON 的貼文列數,133
8,重複貼文 ID 列數,0
9,重複 ig_strong_id 數,1


Missing MP4 posts: 133


,ig_id,post_id,media_id,modified_time_tw,expected_video_filename,has_json,match_status
65,traveggo,4648229675409463,4648229675409463,2025-09-06 22:44:10+08:00,traveggo-20250906224410-4648229675409463.mp4,False,missing_both
302,tzu_888,1215205530776275,1215205530776275,2025-12-09 15:31:21+08:00,tzu_888-20251209153121-1215205530776275.mp4,False,missing_both
1077,wia627,1852338622114597,1852338622114597,2025-09-20 04:58:47+08:00,wia627-20250920045847-1852338622114597.mp4,False,missing_both
1989,zxsdexz,1213717130801192,1213717130801192,2026-01-05 23:09:26+08:00,zxsdexz-20260105230926-1213717130801192.mp4,False,missing_both
1991,0_shufen,1165134668817062,1165134668817062,2025-11-23 04:00:36+08:00,0_shufen-20251123040036-1165134668817062.mp4,False,missing_both
2119,1_shiuan_0,1639657417044170,1639657417044170,2025-08-19 22:25:28+08:00,1_shiuan_0-20250819222528-1639657417044170.mp4,False,missing_both
2352,2uncle987,888601933921971,888601933921971,2025-11-22 01:00:34+08:00,2uncle987-20251122010034-888601933921971.mp4,False,missing_both
3289,1105ya,1051447247181222,1051447247181222,2025-12-12 13:43:16+08:00,1105ya-20251212134316-1051447247181222.mp4,False,missing_both
3501,ackj0301_ifbbpro,1646270259713007,1646270259713007,2026-01-16 03:33:22+08:00,ackj0301_ifbbpro-20260116033322-16462702597130...,False,missing_both
3598,ala780705,1258671062799202,1258671062799202,2025-01-01 14:33:38+08:00,ala780705-20250101143338-1258671062799202.mp4,False,missing_both


## 最終統計表

In [7]:
print("=== SPoD Dataset Overview ===")

print(
    "Total influencers:",
    df_influencer_summary["ig_id"].nunique()
)

print(
    "Influencers with complete data:",
    df_influencer_summary["has_usable_video"].sum()
)

print(
    "Total metadata posts:",
    df_influencer_summary["metadata_post_count"].sum()
)

print(
    "Total MP4 files:",
    df_influencer_summary["mp4_count"].sum()
)

print(
    "Total JSON files:",
    df_influencer_summary["json_count"].sum()
)

print(
    "Total fully matched rows:",
    df_influencer_summary["fully_matched_count"].sum()
)

print(
    "Total video storage (GB):",
    round(
        df_influencer_summary["total_video_size_mb"].sum() / 1024,
        2
    )
)

=== SPoD Dataset Overview ===
Total influencers: 203
Influencers with complete data: 196
Total metadata posts: 19443
Total MP4 files: 19312
Total JSON files: 19116
Total fully matched rows: 19102
Total video storage (GB): 235.73


In [8]:
# 蒐集可能為品牌的 @帳號（僅為候選，不代表已確認品牌；不連線 IG）
# 此 cell 可獨立執行，直接讀取全部原始 Metadata，避免受到前面抽樣／篩選影響。
from pathlib import Path
from collections import Counter, defaultdict
import re
import pandas as pd
from IPython.display import display

brand_metadata_dir = Path("/home/ftp_246/data_5/多模態2026_ig_top200/Metadata_by_Influencer")
brand_profile_path = Path("/home/yingtzudai/Multimodal_SPoD_Video/data/Influencer_Profile.csv")
brand_output_path = Path("/home/yingtzudai/Multimodal_SPoD_Video/data/brands_in_database.csv")
brand_metadata_files = sorted(brand_metadata_dir.rglob("*.csv"))
if not brand_metadata_files:
    raise FileNotFoundError(f"找不到 Metadata CSV：{brand_metadata_dir}")

# 排除名單：Profile 中全部 ig_id、Metadata 的作者帳號，以及來源檔名。
def normalize_brand_handle(value):
    return str(value).strip().lstrip("@＠").lower()

brand_excluded_usernames = {
    normalize_brand_handle(value)
    for value in pd.read_csv(brand_profile_path, dtype=str, keep_default_na=False)["ig_id"]
    if str(value).strip()
}
brand_source_frames = []
for brand_csv_path in brand_metadata_files:
    frame = pd.read_csv(
        brand_csv_path, dtype=str, keep_default_na=False,
        usecols=lambda col: col in {"id", "text", "post_owner.username"},
    )
    required_columns = {"text", "post_owner.username"}
    if not required_columns.issubset(frame.columns):
        raise ValueError(f"{brand_csv_path} 缺少欄位：{required_columns - set(frame.columns)}")
    brand_excluded_usernames.update(
        normalize_brand_handle(value) for value in frame["post_owner.username"] if value.strip()
    )
    if brand_csv_path.name.endswith("-FullVideoInfo.csv"):
        brand_excluded_usernames.add(normalize_brand_handle(
            brand_csv_path.name.removesuffix("-FullVideoInfo.csv")))
    brand_source_frames.append((brand_csv_path, frame))

# 支援半形／全形 @；避免把一般 email 當提及，驗證完整帳號以免截斷長帳號。
brand_mention_pattern = re.compile(r"(?<![a-zA-Z0-9_@.+-])[@＠]([a-zA-Z0-9_.]+)")
brand_handle_pattern = re.compile(r"[a-z0-9_](?:[a-z0-9_.]{0,28}[a-z0-9_])?")
brand_mention_counts = Counter()
brand_post_keys = defaultdict(set)
brand_mentioning_influencers = defaultdict(set)
for brand_csv_path, frame in brand_source_frames:
    for row_number, row in enumerate(frame.to_dict("records"), start=2):
        owner = normalize_brand_handle(row["post_owner.username"])
        post_key = row.get("id", "").strip() or f"{brand_csv_path}:{row_number}"
        for token in brand_mention_pattern.findall(row["text"]):
            handle = token.rstrip(".").lower()
            if (not brand_handle_pattern.fullmatch(handle)
                    or ".." in handle or handle in brand_excluded_usernames):
                continue
            brand_mention_counts[handle] += 1
            brand_post_keys[handle].add(post_key)
            if owner:
                brand_mentioning_influencers[handle].add(owner)

# 每個候選帳號一列；mention_count 是出現次數，post_count 依貼文 ID 去重。
df_brands_in_database = pd.DataFrame([
    {"brand_id": handle,
     "mention_count": count,
     "post_count": len(brand_post_keys[handle]),
     "influencer_count": len(brand_mentioning_influencers[handle])}
    for handle, count in brand_mention_counts.items()
], columns=["brand_id", "mention_count", "post_count", "influencer_count"])
df_brands_in_database = df_brands_in_database.sort_values(
    ["influencer_count", "mention_count", "brand_id"], ascending=[False, False, True],
).reset_index(drop=True)
assert not set(df_brands_in_database["brand_id"]) & brand_excluded_usernames
brand_output_path.parent.mkdir(parents=True, exist_ok=True)
df_brands_in_database.to_csv(brand_output_path, index=False, encoding="utf-8-sig")
print(f"讀取 {len(brand_metadata_files)} 個檔案、{sum(len(f) for _, f in brand_source_frames):,} 筆貼文")
print(f"排除 {len(brand_excluded_usernames)} 個網紅帳號，取得 {len(df_brands_in_database):,} 個品牌候選帳號")
print(f"已儲存：{brand_output_path}")
display(df_brands_in_database.head(20))


讀取 199 個檔案、19,443 筆貼文
排除 203 個網紅帳號，取得 2,564 個品牌候選帳號
已儲存：/home/yingtzudai/Multimodal_SPoD_Video/data/brands_in_database.csv


,brand_id,mention_count,post_count,influencer_count
0,shopee_tw,36,36,29
1,7eleventw,26,26,17
2,watsonstaiwan,16,16,14
3,legotw_official,13,13,13
4,dysonbeauty,12,12,11
5,ubereats_tw,12,11,10
6,netflixtw,22,22,9
7,insta360,20,20,8
8,samsungtaiwan,18,18,8
9,nivea_tw,12,12,8


# 抽樣與統計
**SPoD**：training : validating : testing = 7:1:2

**A simple formula for the calculation of sample size in pilot studies**: 樣本數n = ln(1−γ) / ln(1−p)。其中 p 是研究者在pilot study希望能抓到的「最低問題發生率」，γ 是信心水準。假設 γ = 0.95 p = 0.01，則 n = 299，也就是假設某種問題的發生率是0.01，且每篇post遇到問題的機會彼此獨立，抽查299篇就有至少95%的機會遇到一次這種問題

**目的**
1. 以「人」（person_name）為單位切分，同一人的所有帳號只會出現在單一子集
2. 硬性限制：各子集的貼文比例接近 7:1:2，業配比例與全體一致，且每個子集都有業配樣本
3. 軟性平衡：粉絲級距、網紅類別、發文量的分佈盡量接近全體
4. 從 train 中抽出 299 篇 pilot 樣本

**方法依據**
- 依網紅分組切分：Roberts et al. (2017), *Ecography* 40:913–929（階層結構資料應採 block CV）(Cross-validation strategies for data with temporal, spatial, hierarchical, or phylogenetic structure) 
- Rerandomization：Morgan & Rubin (2012), *Annals of Statistics* 40(2):1263–1282 (Rerandomization to improve covariate balance in experiments)
- 標籤比例分層：Kohavi (1995, IJCAI) (A Study of Cross-Validation and Bootstrap for Accuracy Estimation and Model Selection)；Sechidis et al. (2011, ECML PKDD)(On the Stratification of Multi-label Data)
- 多標籤類別以「各類別盛行率」衡量平衡：概念取自 Sechidis et al. (2011) (On the Stratification of Multi-label Data)
- Pilot 樣本數：Viechtbauer et al. (2015), *J Clin Epidemiol*，p = 0.01、γ = 0.95 → n = 299 (A simple formula for the calculation of sample size in pilot studies)
- 粉絲數、發文量：以樣本內四分位數分為四組（Cochran, 1968, *Biometrics*）

**原則：所有參數都在看到任何模型結果之前固定。**

## 參數設定
所有參數都要在**看到任何模型結果之前**固定，並寫進方法章節。
📖 Morgan & Rubin (2012)：平衡標準必須事先定義，rerandomization 才能保有隨機化的推論效力。

In [62]:
# ---- 路徑 ----
METADATA_DIR = Path("/home/ftp_246/data_5/多模態2026_ig_top200/Metadata_by_Influencer")
PROFILE_CSV  = Path("/home/yingtzudai/Multimodal_SPoD_Video/data/Influencer_Profile.csv")
JSON_DIR     = Path("/home/ftp_246/data_5/多模態2026_ig_top200/Prompt_Description/Fullset_v20")
VIDEO_DIR    = Path("/home/ftp_246/data_5/多模態2026_ig_top200/Video_data_fullest/ig_vedios")
OUT_DIR      = Path("/home/yingtzudai/Multimodal_SPoD_Video/data/split")

# ---- 切分 ----
SEED   = 2026
SPLITS = ["train", "val", "test"]
TARGET = np.array([0.7, 0.1, 0.2])        # SPoD 的 7:1:2，以「貼文數」計

# ---- 粉絲規模：樣本內粉絲數四分位數分組 ----
N_TIER_GROUPS   = 4
N_VOL_GROUPS    = 4
MIN_PER_STRATUM = 3      # 人數 < 3 的「類別」無法分到三個子集，不納入平衡分數

# ---- Rerandomization ----
N_DRAWS      = 200_000    # 隨機切分的次數
ACCEPT_RATE  = 0.05      # 只接受平衡分數最好的 5%（Morgan & Rubin 的 p_a）
MAX_POST_DEV = 0.02      # 硬性：各子集貼文佔比與目標差距 ≤ 2 個百分點
MAX_SPON_DEV = 0.02      # 硬性：各子集業配比例與全體差距 ≤ 2 個百分點
W = dict(posts=1.0, spon=1.0, tier=0.5, cat=0.5, vol=0.5)   # 軟性平衡權重
# ---- 貼文文字 ----
EXCLUDE_EMPTY_TEXT = True   # True：排除 text 為空的貼文；False：只標記、不排除

# ---- Pilot（Viechtbauer et al. 2015）----
PILOT_P, PILOT_CONF = 0.01, 0.95

## 檢查網紅Profile
- 切分單位是**人**（`person_name`），同一人的所有帳號必須落在同一個子集。
  📖 Roberts et al. (2017)：資料有階層結構時，以群組為單位切分，才能避免低估預測誤差。
- `AsiaKOL_category` 可能是 `[搞笑，藝術]` 這種多值格式，所以解析成 list（**多標籤**）。
- ⚠️ 這裡假設 profile 的 `ig_id` 等於 metadata 的 `post_owner.username`。Cell 9 會檢查對不上的筆數。

In [63]:
profile = pd.read_csv(PROFILE_CSV, dtype=str)          # dtype=str：避免 id 被轉成 float
profile.columns = profile.columns.str.strip()
for c in ["person_name", "ig_id", "ig_strong_id"]:
    profile[c] = profile[c].str.strip()
profile["Followers"] = pd.to_numeric(
    profile["Followers"].str.replace(",", "", regex=False), errors="coerce")

def parse_categories(x):
    """'[搞笑，藝術]'、\"['搞笑', '藝術']\"、'搞笑,藝術' → ['搞笑', '藝術']"""
    if pd.isna(x):
        return []
    s = re.sub(r"[\[\]'\"]", "", str(x))
    return sorted({c.strip() for c in re.split(r"[,，、/;；]", s) if c.strip()})

profile["cat_list"] = profile["AsiaKOL_category"].map(parse_categories)
acct2person = dict(zip(profile["ig_id"], profile["person_name"]))

n_acc = profile.groupby("person_name")["ig_id"].nunique()
print(f"帳號數: {profile['ig_id'].nunique()}　人數: {profile['person_name'].nunique()}")
print(f"多帳號網紅: {(n_acc > 1).sum()} 位")
display(n_acc[n_acc > 1].sort_values(ascending=False))
print("Followers 無法解析的帳號:", profile["Followers"].isna().sum())

帳號數: 203　人數: 200
多帳號網紅: 2 位


person_name
好味小姐      3
阿啾繪圖同萌    2
Name: ig_id, dtype: int64

Followers 無法解析的帳號: 0


## 組出檔名
規則：`{post_owner.username}-{modified_time_tw 的年月日時分秒}-{media_id}`

⚠️ 兩個陷阱：
1. `"2025-04-29 03:06:06+00:00"` 不能「把所有數字抓出來」，否則時區的 `0000` 也會被接上去。這裡只取前面的年月日時分秒。
2. `media_id` 有 16 位以上，用預設方式讀 CSV 會被轉成 float 而失去精度，**一定要 `dtype=str`**。

In [64]:
DT_RE = re.compile(r"^\s*(\d{4})-(\d{2})-(\d{2})[ T](\d{2}):(\d{2}):(\d{2})")

def ts_digits(s):
    m = DT_RE.match(str(s))
    return "".join(m.groups()) if m else None

# 單元測試
assert ts_digits("2025-04-29 03:06:06+00:00") == "20250429030606"
assert ts_digits("2025-04-29T03:06:06.123+08:00") == "20250429030606"

## 讀取 metadata，比對 mp4 / json 是否存在
先把資料夾裡的所有檔名讀成 set，再做比對，比逐一呼叫 `Path.exists()` 快很多（尤其在網路磁碟上）。

In [65]:
frames = []
for f in sorted(METADATA_DIR.glob("*-FullVideoInfo.csv")):
    df = pd.read_csv(f, dtype=str)
    df["source_csv"] = f.name
    frames.append(df)
meta = pd.concat(frames, ignore_index=True)

meta["username"] = meta["post_owner.username"].str.strip()
meta["media_id"] = meta["media_id"].str.strip()
meta["ts"]       = meta["modified_time_tw"].map(ts_digits)
meta["stem"]     = meta["username"] + "-" + meta["ts"] + "-" + meta["media_id"]
print(f"metadata 總列數: {len(meta)}　時間無法解析: {meta['ts'].isna().sum()}")

existing_mp4  = {p.stem for p in VIDEO_DIR.glob("*/*.mp4")}
existing_json = {p.stem for p in JSON_DIR.glob("*/*.json")}
meta["mp4_ok"]     = meta["stem"].isin(existing_mp4)
meta["json_ok"]    = meta["stem"].isin(existing_json)
meta["in_profile"] = meta["username"].isin(acct2person)
# 標記 text 為空的貼文：NaN、空字串，或只有空白／換行／零寬字元
assert "text" in meta.columns, "metadata 缺少 text 欄位"
INVISIBLE = r"[\s\u200b\u200c\u200d\u2060\ufeff]+"
meta["text_empty"] = (meta["text"].isna()
                      | (meta["text"].fillna("").str.replace(INVISIBLE, "", regex=True) == ""))

display(meta[["mp4_ok", "json_ok", "in_profile", "text_empty"]].sum().to_frame("符合筆數"))
print("profile 中找不到的 username:", sorted(meta.loc[~meta["in_profile"], "username"].unique())[:20])

metadata 總列數: 19443　時間無法解析: 0


,符合筆數
mp4_ok,19310
json_ok,19114
in_profile,19443
text_empty,44


profile 中找不到的 username: []


## 讀取標籤 `isEndorsement`
位置：`high_inference → endorsement_analysis → isEndorsement`
同時檢查 json 內的 `videoName` 是否等於預期的 mp4 檔名，以防檔名對錯。

⚠️ 這個 json 的其他欄位**不能當模型特徵**，例如 `endorsementCues`、`endorsementObject`、`text_narrative_log`。
它們和標籤來自同一次分析，等於直接洩漏答案（對應 SPoD 刪除 #ad 等揭露關鍵字的作法）。

**text 為空的貼文**
- 判定標準：`text` 為 NaN、空字串，或只含空白／換行／零寬字元
- 由參數 `EXCLUDE_EMPTY_TEXT` 決定排除或只標記；被排除的清單另存為 `excluded_empty_text.csv`
- 此決定在切分前、看到任何模型結果前做出

In [66]:
def read_label(json_path, expected_video):
    try:
        with open(json_path, encoding="utf-8") as fh:
            d = json.load(fh)
        v = d["high_inference"]["endorsement_analysis"]["isEndorsement"]
    except (OSError, json.JSONDecodeError, KeyError, TypeError):
        return np.nan, "read_error"
    if d.get("videoName") != expected_video:
        return np.nan, "name_mismatch"
    if isinstance(v, bool):
        return int(v), "ok"
    if isinstance(v, str) and v.strip().lower() in {"true", "false"}:
        return int(v.strip().lower() == "true"), "ok"
    return np.nan, "bad_value"

cand = meta[meta["mp4_ok"] & meta["json_ok"] & meta["in_profile"]].copy()
res = [read_label(JSON_DIR / u / f"{s}.json", f"{s}.mp4")
       for u, s in zip(cand["username"], cand["stem"])]
cand["label"]        = [r[0] for r in res]
cand["label_status"] = [r[1] for r in res]
display(cand["label_status"].value_counts())

valid = cand[cand["label_status"] == "ok"].drop_duplicates("media_id")

# 標出 text 為空的貼文，另外存一份清單備查
excluded_empty_text = valid[valid["text_empty"]].copy()
print(f"text 為空的貼文: {len(excluded_empty_text)} 篇"
      f"（業配 {int(excluded_empty_text['label'].sum())} 篇）"
      f"，涉及 {excluded_empty_text['username'].nunique()} 個帳號")

if EXCLUDE_EMPTY_TEXT:
    valid = valid[~valid["text_empty"]]
    print("→ 已排除")
else:
    print("→ 只標記，未排除")

posts = (valid
         .assign(label=lambda d: d["label"].astype(int),
                 person_id=lambda d: d["username"].map(acct2person))
         .reset_index(drop=True))

print(f"可用貼文: {len(posts)}")
print(f"帳號: {posts['username'].nunique()}　人: {posts['person_id'].nunique()}")
print(f"整體業配比例: {posts['label'].mean():.2%}")

label_status
ok               19102
name_mismatch       12
Name: count, dtype: int64

text 為空的貼文: 30 篇（業配 8 篇），涉及 19 個帳號
→ 已排除
可用貼文: 19072
帳號: 196　人: 194
整體業配比例: 47.91%


## 6. 建立「人」層級的屬性表
| 屬性 | 定義 | 理由 |
|---|---|---|
| `max_followers` | 該人所有帳號中，粉絲數**最多**的帳號 | 加總會重複計算同一批受眾；品牌通常依主帳號評估合作（研究者自訂的規則） |
| `tier` | 依 `max_followers` 做樣本內**四分位數**分組（Q1–Q4） | 只作為分層變數；4 層可保留連續變數的大部分資訊（Cochran, 1968） |
| `cats` | 所有帳號類別的**聯集**（多標籤） | 📖 Sechidis et al. (2011)：多標籤資料的平衡應逐一標籤處理 |
| `vol_bin` | 依貼文數做樣本內**四分位數**分組（Q1–Q4） | 發文量大的網紅會主導子集的貼文比例 |

In [67]:
person = posts.groupby("person_id").agg(n_posts=("label", "size"), n_spon=("label", "sum"))
person["max_followers"] = profile.groupby("person_name")["Followers"].max().reindex(person.index)
person["cats"] = (profile.groupby("person_name")["cat_list"]
                  .agg(lambda L: sorted(set().union(*L))).reindex(person.index))

# 粉絲數：樣本內四分位數分組
person["tier"] = pd.qcut(person["max_followers"].rank(method="first"), N_TIER_GROUPS,
                         labels=[f"Q{i}" for i in range(1, N_TIER_GROUPS + 1)]).astype(str)

# 發文量：樣本內四分位數分組
person["vol_bin"] = pd.qcut(person["n_posts"].rank(method="first"), N_VOL_GROUPS,
                            labels=[f"Q{i}" for i in range(1, N_VOL_GROUPS + 1)]).astype(str)

# 記錄每組的實際範圍（論文報告與存檔用）
tier_ranges = (person.groupby("tier")["max_followers"]
               .agg(["count", "min", "max"]).rename(columns={"count": "人數"}))
vol_ranges  = (person.groupby("vol_bin")["n_posts"]
               .agg(["count", "min", "max"]).rename(columns={"count": "人數"}))
display(tier_ranges.rename_axis("粉絲數四分位"))
display(vol_ranges.rename_axis("發文量四分位"))

# 類別 multi-hot；人數 < 3 的類別無法平衡，不納入分數
cat_mh = (pd.DataFrame([{c: 1 for c in L} for L in person["cats"]], index=person.index)
          .fillna(0).astype(int))
cat_counts = cat_mh.sum().sort_values(ascending=False)
display(cat_counts.to_frame("人數"))
cat_mh = cat_mh.loc[:, cat_counts.reindex(cat_mh.columns) >= MIN_PER_STRATUM]

,人數,min,max
粉絲數四分位,,,
Q1,49,8713,138411
Q2,48,139725,238862
Q3,48,240547,432782
Q4,49,433921,2450201


,人數,min,max
發文量四分位,,,
Q1,49,1,39
Q2,48,40,68
Q3,48,68,127
Q4,49,128,663


,人數
趣味搞笑,53
旅遊,37
表演藝術,35
美食料理,31
遊戲電玩,28
美妝保養,25
時尚潮流,24
知識教育,15
寵物,14
家庭母嬰,13


## 7. 定義「一次隨機切分」與「平衡分數」

**隨機切分**：把人隨機排序，依累積貼文數填入 train（前 70%）→ val（10%）→ test（20%）。

**硬性門檻**（不符合就直接淘汰）：
- 各子集貼文佔比與 7:1:2 的差距 ≤ `MAX_POST_DEV`
- 各子集業配比例與全體的差距 ≤ `MAX_SPON_DEV`
  📖 Kohavi (1995)；Sechidis et al. (2011)。另外，AP / MAP 的隨機基準等於正例比例，比例不同的子集，指標就不在同一把尺上
- 每個子集都要同時有業配與非業配貼文
  📖 Sechidis et al. (2011)：測試折中沒有正例，recall、AP 等指標會無法計算

**軟性平衡分數**（越小越好）：
- 粉絲級距、發文量：各子集分佈與全體分佈的差距加總 (total variation distance)
- 類別：每個類別在各子集的盛行率與全體的平均絕對差（label-wise，📖 Sechidis et al. 2011, Section 2 與 Section 4.2 的 LD 指標）

> 註：Morgan & Rubin (2012) 對連續共變數建議使用 Mahalanobis distance。這裡的共變數都是類別型變數，所以改用上述自訂指標。論文中要說明這是改編，而非原作的設定。

In [68]:
# =====================================================================
# 準備工作：把「人」層級的資料轉成 numpy 陣列
# ---------------------------------------------------------------------
# 之後要重複切分 20 萬次，每次都從 pandas 取資料會很慢。
# 所以先把每位網紅的資料轉成陣列，第 i 個位置永遠代表同一位網紅。
# =====================================================================
ids     = person.index.to_numpy()                 # 每位網紅的 person_id（順序固定）
n_posts = person["n_posts"].to_numpy(float)       # 每位網紅的貼文數
n_spon  = person["n_spon"].to_numpy(float)        # 每位網紅的業配貼文數
TOTAL   = n_posts.sum()                           # 全部貼文數（19,102）
G_SPON  = n_spon.sum() / TOTAL                    # 全體業配比例（約 47.88%），之後當作比較基準
TIER_OH = pd.get_dummies(person["tier"]).to_numpy(float)   # 粉絲四分位的 one-hot 表
                                                             # 例：某人在 Q2 → [0, 1, 0, 0]
VOL_OH  = pd.get_dummies(person["vol_bin"]).to_numpy(float)  # 發文量四等分的 one-hot 表
                                                             # 例：某人是 Q1 → [1, 0, 0, 0]
CAT_MH  = cat_mh.to_numpy(float)                  # 類別的 multi-hot 表（一人可同時有多個 1）
                                                  # 例：[搞笑, 旅遊] → 搞笑欄=1、旅遊欄=1，其他欄=0
N       = len(ids)                                # 網紅人數（194）


def random_split(rng):
    """
    【目的】產生「一次」隨機切分：決定每位網紅要去 train、val 還是 test。

    【做法】想像 194 位網紅隨機排成一列，從頭開始依序走進教室：
            累積的貼文數還沒到全體的 70% → 進 train
            累積到 70%～80% 之間        → 進 val
            累積超過 80%                → 進 test
            以「貼文數」而不是「人數」計算比例，因為 SPoD 的 7:1:2 指的是貼文
            （SPoD Section 4.2）。這個填滿方式是研究者自訂的，沒有文獻出處。

    【輸入】rng：隨機數產生器（固定 seed，才能重現同樣的結果）
    【輸出】assign：長度 N 的陣列，第 i 個值代表第 i 位網紅分到哪裡
            0 = train、1 = val、2 = test
    """
    # 1. 把 0～193 的編號隨機洗牌，等於讓網紅隨機排隊
    order = rng.permutation(N)

    # 2. 計算每個人「走進教室那一刻」，前面的人已經累積了多少比例的貼文
    #    例：排隊順序的貼文數是 [100, 50, 200, ...]，全體 19,102 篇
    #        第 1 人進門時累積 0      → 0.000
    #        第 2 人進門時累積 100    → 0.005
    #        第 3 人進門時累積 150    → 0.008 ...
    #    （cumsum 是累加；[:-1] 加上開頭補 0，代表「不含自己」的累積量）
    start_frac = np.concatenate([[0.0], np.cumsum(n_posts[order])[:-1]]) / TOTAL

    # 3. 教室的分界線：train 到 0.7 為止，val 到 0.8 為止
    bounds = np.cumsum(TARGET)[:-1]                     # [0.7, 0.8]

    # 4. 依照進門時的累積比例，決定每個人進哪一間：
    #      start_frac < 0.7        → 0（train）
    #      0.7 ≤ start_frac < 0.8  → 1（val）
    #      start_frac ≥ 0.8        → 2（test）
    #    注意：判斷的是「進門那一刻」。所以如果某人在 0.69 進門、又帶了很多貼文，
    #    他會整個人留在 train，讓 train 超過 70%。
    #    這就是為什麼後面 evaluate() 還要檢查比例有沒有偏太多。
    assign = np.empty(N, dtype=int)
    assign[order] = np.searchsorted(bounds, start_frac, side="right")   # 0/1/2
    #    assign[order] = ... 的作用是把結果「放回原本的編號位置」，
    #    讓 assign[i] 永遠對應 ids[i] 這位網紅

    return assign


def tvd_block(onehot, assign):
    """
    【目的】衡量三個子集的「圓餅圖」和全體的圓餅圖差多少。
            用於粉絲四分位（TIER_OH）和發文量四分位（VOL_OH）。

    【做法】Total Variation Distance（TVD）：
            把每一塊的比例差距取絕對值、加總，再除以 2。
            白話：「要把多少比例的人換到別的組，兩張圓餅圖才會一模一樣。」
            例：全體 Q1～Q4 各約 25%，val 是 [26.9%, 19.2%, 26.9%, 26.9%]
                → TVD ≈ 0.055，也就是 val 裡約 5.5% 的人需要換組
            TVD 是統計上的標準距離；選用它是研究者的決定，沒有特定論文出處。

    【輸入】onehot：one-hot 表（每列一人、每欄一個組別）
            assign：random_split() 的結果
    【輸出】一個數字 = train、val、test 三個 TVD 的總和
            0 代表三個子集都和全體完全一樣；數字越大越不平衡
    """
    # 全體的圓餅圖：每一欄的平均 = 該組人數佔全體的比例
    #   例：[0.253, 0.247, 0.247, 0.253]
    overall = onehot.mean(0)

    # 對 train(0)、val(1)、test(2) 各算一次 TVD，再加總
    #   onehot[assign == s]  → 只取分到子集 s 的那些人
    #   .mean(0)             → 這個子集自己的圓餅圖
    #   np.abs(... - overall).sum() * 0.5 → TVD
    return sum(0.5 * np.abs(onehot[assign == s].mean(0) - overall).sum() for s in range(3))


def evaluate(assign):
    """
    【目的】幫「一次切分」打分數，分成兩關：
            第一關 hard_ok：硬性門檻，任一條不符合就淘汰
            第二關 score  ：平衡分數，越小代表三個子集越像全體

    【輸入】assign：random_split() 的結果
    【輸出】(hard_ok, score, parts)
            hard_ok：True / False，有沒有通過硬性門檻
            score  ：加權後的總平衡分數
            parts  ：五個分項各自的分數（方便檢查是哪一項不平衡）
    """
    # ---------- 計算每個子集的貼文數與業配數 ----------
    # bincount 加上 weights：依照分組，把每個人的貼文數加總
    #   例：posts_s = [13,371, 1,912, 3,819] → train / val / test 各有幾篇貼文
    posts_s = np.bincount(assign, weights=n_posts, minlength=3)
    spon_s  = np.bincount(assign, weights=n_spon,  minlength=3)

    # 保險：如果某個子集一個人都沒有，後面會除以 0，直接判定不合格
    if (posts_s == 0).any():
        return False, np.inf, {}

    share = posts_s / TOTAL    # 各子集的貼文佔比，例：[0.700, 0.100, 0.200]
    rate  = spon_s / posts_s   # 各子集的業配比例，例：[0.478, 0.485, 0.474]

    # ---------- 第一關：硬性門檻（全部都要成立才算通過） ----------
    hard_ok = (
        # ① 貼文佔比與 7:1:2 的差距 ≤ 2 個百分點
        #    依據：SPoD Section 4.2（7:1:2）；Sechidis Section 4.2 的 ED 指標概念
        np.abs(share - TARGET).max() <= MAX_POST_DEV

        # ② 各子集業配比例與全體的差距 ≤ 2 個百分點
        #    依據：SPoD Section 4.2 "the ratios… are the same"；Kohavi (1995)；
        #          Saito & Rehmsmeier (2015)：AP 的隨機基準 = 正例比例，比例不同就不能比較
        and np.abs(rate - G_SPON).max() <= MAX_SPON_DEV

        # ③ 每個子集都至少有 1 篇業配
        # ④ 每個子集都至少有 1 篇非業配
        #    依據：Sechidis Section 2 最後一段、Section 4.2 的 FZ/FLZ：
        #          沒有正例（或沒有負例），AP 就算不出來
        and (spon_s > 0).all()
        and (posts_s - spon_s > 0).all()
    )
    # （「2 個百分點」是研究者自訂的數字，沒有文獻出處）

    # ---------- 第二關：五個分項的平衡分數 ----------
    overall_cat = CAT_MH.mean(0)   # 全體中，每個類別的盛行率（例：搞笑 27.3%、旅遊 19.1%…）

    parts = dict(
        # 貼文佔比與 7:1:2 的差距總和
        posts=np.abs(share - TARGET).sum(),

        # 業配比例與全體的差距總和
        spon=np.abs(rate - G_SPON).sum(),

        # 粉絲四分位的圓餅圖差距（TVD）
        tier=tvd_block(TIER_OH, assign),

        # 類別盛行率的差距：
        #   對每個子集，逐一比較「每個類別的盛行率」與全體的差距，再取平均；
        #   最後把三個子集加總
        #   例：寵物類全體 7.2%、val 11.5% → 這一格差 4.3 個百分點
        #   依據：Sechidis Section 2 第三段（逐一標籤處理）與 Section 4.2 的 LD 指標；
        #         原論文比較的是「正例：負例比值」，這裡改用盛行率，比較直觀，
        #         也不會在負例很少時數值暴增
        cat=sum(np.abs(CAT_MH[assign == s].mean(0) - overall_cat).mean() for s in range(3)),

        # 發文量四等分的圓餅圖差距（TVD）
        vol=tvd_block(VOL_OH, assign),
    )

    # 加權總分：W = dict(posts=1.0, spon=1.0, tier=0.5, cat=0.5, vol=0.5)
    #   貼文比例和業配比例會直接影響評估指標，所以權重較高；
    #   其他三項只要求「盡量像」。權重是研究者事先決定的，沒有文獻出處。
    #   依據：Morgan & Rubin Section 2.1，平衡標準可以是任何函數，
    #         但必須在切分之前就定好
    return hard_ok, sum(W[k] * v for k, v in parts.items()), parts

## 8. Rerandomization
📖 Morgan & Rubin (2012) 的程序是：隨機分配 → 檢查是否符合事先定義的平衡標準 → 不符合就重抽，直到符合為止。

實作上等價於：
1. 抽 `N_DRAWS` 次，保留通過硬性門檻的切分
2. 接受標準 = 平衡分數最好的 `ACCEPT_RATE`（對應論文的接受機率 p_a）
3. **從接受集合中隨機選一個**，而不是直接挑分數最好的那一個

第 3 步很重要：直接挑「最好的一個」會讓切分變成確定性的結果，就失去了隨機化的性質。

In [69]:
rng = np.random.default_rng(SEED)
passed = []
for _ in range(N_DRAWS):
    a = random_split(rng)
    ok, sc, _ = evaluate(a)
    if ok:
        passed.append((sc, a))

print(f"通過硬性門檻: {len(passed)} / {N_DRAWS}（{len(passed)/N_DRAWS:.2%}）")
assert passed, "沒有任何切分通過硬性門檻：請放寬 MAX_POST_DEV / MAX_SPON_DEV，並在論文中記錄理由"

scores    = np.array([s for s, _ in passed])
threshold = np.quantile(scores, ACCEPT_RATE)
accepted  = [a for s, a in passed if s <= threshold]
assign    = accepted[rng.integers(len(accepted))]
_, chosen_score, chosen_parts = evaluate(assign)

print(f"接受集合: {len(accepted)} 個　門檻: {threshold:.4f}　選中分數: {chosen_score:.4f}")
print({k: round(float(v), 4) for k, v in chosen_parts.items()})

person["split"] = np.array(SPLITS)[assign]
posts["split"]  = posts["person_id"].map(person["split"])

通過硬性門檻: 8867 / 200000（4.43%）
接受集合: 444 個　門檻: 0.2584　選中分數: 0.2510
{'posts': 0.0076, 'spon': 0.0304, 'tier': 0.2134, 'cat': 0.0702, 'vol': 0.1424}


## 9. 檢查洩漏與平衡
這三張表可以直接放進論文附錄，作為切分品質的證據。

In [70]:
# 洩漏檢查：同一人、同一帳號只能出現在一個子集
assert posts.groupby("person_id")["split"].nunique().max() == 1
assert posts.groupby("username")["split"].nunique().max() == 1
assert posts["split"].notna().all()

rows = []
for s in SPLITS:
    P, Q = posts[posts["split"] == s], person[person["split"] == s]
    rows.append({"split": s, "人數": len(Q), "帳號數": P["username"].nunique(),
                 "貼文數": len(P), "貼文佔比": len(P) / len(posts),
                 "業配數": int(P["label"].sum()), "業配比例": P["label"].mean()})
summary = pd.DataFrame(rows).set_index("split")
display(summary.style.format({"貼文佔比": "{:.2%}", "業配比例": "{:.2%}"}))

display(pd.crosstab(person["tier"], person["split"], normalize="columns")[SPLITS]
        .round(3).rename_axis("粉絲數四分位（人數比例）"))
display(pd.crosstab(person["vol_bin"], person["split"], normalize="columns")[SPLITS]
        .round(3).rename_axis("發文量（人數比例）"))

cat_table = pd.DataFrame({s: cat_mh[person["split"] == s].mean() for s in SPLITS})
cat_table["all"] = cat_mh.mean()
display(cat_table.round(3).rename_axis("類別盛行率"))

,人數,帳號數,貼文數,貼文佔比,業配數,業配比例
split,,,,,,
train,136,138,13379,70.15%,6338,47.37%
val,24,24,1951,10.23%,958,49.10%
test,34,34,3742,19.62%,1842,49.23%


split,train,val,test
粉絲數四分位（人數比例）,,,
Q1,0.257,0.292,0.206
Q2,0.213,0.250,0.382
Q3,0.250,0.250,0.235
Q4,0.279,0.208,0.176


split,train,val,test
發文量（人數比例）,,,
Q1,0.243,0.250,0.294
Q2,0.257,0.250,0.206
Q3,0.250,0.292,0.206
Q4,0.250,0.208,0.294


,train,val,test,all
類別盛行率,,,,
3C科技,0.022,0.000,0.088,0.031
知識教育,0.081,0.083,0.059,0.077
醫療健康,0.022,0.042,0.000,0.021
表演藝術,0.206,0.125,0.118,0.180
趣味搞笑,0.279,0.167,0.324,0.273
美食料理,0.154,0.167,0.176,0.160
旅遊,0.162,0.292,0.235,0.191
時尚潮流,0.125,0.125,0.118,0.124
美妝保養,0.132,0.125,0.118,0.129


## Pilot 樣本（只從 train 抽）
📖 Viechtbauer et al. (2015)：n = ⌈ln(1−γ) / ln(1−p)⌉，p = 0.01、γ = 0.95 → n = 299
- 以**貼文**為單位做簡單隨機抽樣，維持自然的業配比例
- 只從 train 抽，pilot 階段的任何調整都不會碰到 test
- 用途**僅限**偵測資料與管線問題，不用來估計模型表現

In [71]:
pilot_n = math.ceil(math.log(1 - PILOT_CONF) / math.log(1 - PILOT_P))
train_posts = posts[posts["split"] == "train"]
assert pilot_n <= len(train_posts)

pilot = train_posts.sample(n=pilot_n, random_state=SEED + 1)
print(f"pilot n = {pilot_n}　業配 {pilot['label'].sum()} 篇（{pilot['label'].mean():.2%}）"
      f"　涵蓋 {pilot['person_id'].nunique()} 位網紅")

pilot n = 299　業配 136 篇（45.48%）　涵蓋 97 位網紅


## 存檔
連同所有參數與 rerandomization 的結果一起存，確保可以重現。

In [72]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
keep = ["media_id", "id", "username", "person_id", "stem", "label", "split"]

posts[keep].to_csv(OUT_DIR / "post_split.csv", index=False, encoding="utf-8-sig")
pilot[keep].to_csv(OUT_DIR / "pilot_posts.csv", index=False, encoding="utf-8-sig")
(person.assign(cats=person["cats"].map("|".join))
       .to_csv(OUT_DIR / "person_split.csv", encoding="utf-8-sig"))
summary.to_csv(OUT_DIR / "split_summary.csv", encoding="utf-8-sig")
excluded_empty_text[["media_id", "id", "username", "stem", "label"]].to_csv(
    OUT_DIR / "excluded_empty_text.csv", index=False, encoding="utf-8-sig")

config = dict(seed=SEED, exclude_empty_text=EXCLUDE_EMPTY_TEXT, n_empty_text=len(excluded_empty_text), 
              target=TARGET.tolist(), tier_method="sample quartiles of max followers per person",
              n_tier_groups=N_TIER_GROUPS,
              tier_ranges={q: [int(r["min"]), int(r["max"])] for q, r in tier_ranges.iterrows()},
              vol_method="sample quartiles of post count per person",
              n_vol_groups=N_VOL_GROUPS,
              vol_ranges={q: [int(r["min"]), int(r["max"])] for q, r in vol_ranges.iterrows()},
              min_per_stratum=MIN_PER_STRATUM,
              n_draws=N_DRAWS, accept_rate=ACCEPT_RATE,
              max_post_dev=MAX_POST_DEV, max_spon_dev=MAX_SPON_DEV, weights=W,
              n_passed=len(passed), n_accepted=len(accepted),
              threshold=float(threshold), chosen_score=float(chosen_score),
              pilot_p=PILOT_P, pilot_conf=PILOT_CONF, pilot_n=pilot_n)
with open(OUT_DIR / "split_config.json", "w", encoding="utf-8") as fh:
    json.dump(config, fh, ensure_ascii=False, indent=2)
print("已存檔至", OUT_DIR.resolve())

已存檔至 /home/yingtzudai/Multimodal_SPoD_Video/data/split
